# Encrypted RAG Chatbot with CyborgDB

> **Last validated: 2026-10-06 | CyborgDB v0.18 (disk-backed)**

CyborgDB 0.18 runs as a single-node service with an on-disk encrypted index. Everything in this notebook executes inside the Colab kernel.

## 🚀 Quick Start Guide

This notebook is designed to run on **Google Colab** and will automatically set up everything you need!

---

### Step 1: Add API Keys

You'll need an **OpenAI API key** (get it from https://platform.openai.com/api-keys) to use a hosted LLM for demo purposes.

You can set it as the `OPENAI_API_KEY` environment variable, or be prompted when you run the notebook.

---

### Step 2: Run All Cells

Click **Runtime → Run all** and the notebook will:
- ✅ Install all Python dependencies
- ✅ Start the CyborgDB service
- ✅ Launch the Gradio chatbot interface

**Note:** On first run, the kernel restarts automatically after installing packages (this prevents version conflicts). If it doesn't restart on its own, click **Runtime → Restart and run all** and you're set.

---

### Step 3: Open the Public URL

When the last cell finishes running, you'll see a **public Gradio URL** (e.g., `https://xxxxx.gradio.live`).

**Click the URL** to open the chatbot interface in a new tab, then:
1. Upload PDF or TXT documents
2. Click "Create Encrypted Vector Store"
3. Pick a retrieval mode (Hybrid, Vector, or BM25) and start asking questions about your documents!

All vector embeddings are encrypted end-to-end.

In [ ]:
# 0. Install dependencies and restart automatically if needed

# Pinned versions (validated 2026-10-06 on Google Colab, Python 3.13).
# This notebook uses only the standalone langchain integration packages
# (community / text-splitters / huggingface / openai). It does NOT import
# anything from the `langchain` namespace itself.
%pip install \
    "numpy==2.4.3" \
    "getpass4==0.0.14.1" \
    "scipy==1.17.1" \
    "scikit-learn==1.9.0" \
    "transformers==5.11.0" \
    "sentence-transformers==5.5.1" \
    "langchain-community==0.4.2" \
    "langchain-text-splitters==1.1.2" \
    "langchain-huggingface==1.2.2" \
    "langchain-openai==1.3.0" \
    "pypdf==6.13.2" \
    "openai==2.41.1" \
    "gradio==6.17.3" \
    "cryptography==48.0.1" \
    "cyborgdb==0.18.0" \
    "cyborgdb-service==0.18.0"

# Colab imports some packages (numpy in particular) at startup. If the install above replaced
# one that was already loaded, restart so the next cells don't mix old and new versions in memory.
import sys, importlib.metadata
_dists = {"numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn", "transformers": "transformers",
          "huggingface_hub": "huggingface_hub", "tokenizers": "tokenizers", "openai": "openai"}
_stale = [d for m, d in _dists.items()
          if m in sys.modules and sys.modules[m].__version__ != importlib.metadata.version(d)]
if _stale:
    print(f"🔄 Restarting the runtime to load the updated {', '.join(_stale)}. Then run all cells again.")
    get_ipython().kernel.do_shutdown(restart=True)
else:
    print("✅ Dependencies ready.")

In [ ]:
# 1. Configure API keys and environment variables

import os
import getpass

# Single-node disk-backed encrypted index.
CYBORGDB_DB_TYPE = 'disk'

# Get OpenAI API key from environment variable or prompt
OPENAI_API_KEY = (os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API Key: ")).strip()
if not OPENAI_API_KEY.isascii():
    raise ValueError("OPENAI_API_KEY contains non-ASCII characters (often a smart dash or quote picked up "
                     "while copying). Copy the key directly from the OpenAI dashboard and re-run this cell.")

# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["CYBORGDB_DB_TYPE"] = CYBORGDB_DB_TYPE
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "false"

In [ ]:
# 2. Start CyborgDB service and wait for it to be ready

import subprocess
import time
import requests

# Start service
print("🔄 Starting CyborgDB service...")
log_file = "cyborgdb_service.log"

# Only pass the env vars the service needs (avoids inheriting unwanted shell vars)
service_env = {
    "PATH": os.environ["PATH"],
    "HOME": os.environ["HOME"],
    "CYBORGDB_DB_TYPE": "disk",
}

process = subprocess.Popen(
    "cyborgdb-service",
    stdout=open(log_file, "w"),
    stderr=subprocess.STDOUT,
    env=service_env
)

# Wait for startup
for i in range(60):
    try:
        response = requests.get("http://localhost:8000/v1/health", timeout=1)
        if response.status_code == 200:
            print("\n✅ CyborgDB service started!")
            break
    except:
        pass
    time.sleep(1)
    if i % 5 == 0 and i > 0:
        print(f"   Waiting... ({i}/60)")
else:
    print("\n❌ Service failed to start. Logs:")
    !tail -30 {log_file}
    raise RuntimeError("Failed to start service")

In [ ]:
# 3. Main Application Code

import os
import pathlib
import base64
import html
import re
import uuid
import numpy as np
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import langchain_huggingface  # noqa: F401 — required at runtime for CyborgVectorStore's HF embeddings
import langchain_openai
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import cyborgdb
from cyborgdb.integrations.langchain import CyborgVectorStore
import gradio as gr

print("✅ Libraries imported")

# Settings
EMBEDDING_MODEL_ID = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384
OPENAI_MODEL = "gpt-4o-mini"
OPENAI_URL = None
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
CYBORGDB_HOST = "http://localhost:8000"
CYBORGDB_KEYS = {}
VECTORSTORE_STORE = {}
CHAT_HISTORY_STORE = {}   # session_id -> list[(role, content)]
LLM_STORE = {}            # session_id -> ChatOpenAI
LAST_QUERY_INFO = {}
# Hybrid blend weight per mode: 1 = pure vector, 0 = pure BM25 keyword search.
RETRIEVAL_MODES = {"Vector": 1.0, "Hybrid": 0.5, "BM25": 0.0}
MOST_RECENT_PROMPT = None

print("✅ Settings configured")

# Functions
def split_documents(file_path: str) -> list:
    file = pathlib.Path(file_path)
    if not file.exists():
        raise ValueError("File not found")
    loader_cls = PyPDFLoader if file.suffix == ".pdf" else TextLoader
    pages = loader_cls(str(file)).load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    return splitter.split_documents(pages)

def generate_encryption_key() -> bytes:
    """Generate a 256-bit AES-GCM key."""
    return AESGCM.generate_key(bit_length=256)

def encrypt_chunk(text: str, key: bytes) -> str:
    aesgcm = AESGCM(key)
    nonce = os.urandom(12)
    ciphertext = aesgcm.encrypt(nonce, text.encode(), None)
    return base64.b64encode(nonce + ciphertext).decode()

def load_vectorstore(session_id: str) -> CyborgVectorStore:
    if session_id in VECTORSTORE_STORE:
        return VECTORSTORE_STORE[session_id]
    if session_id not in CYBORGDB_KEYS:
        CYBORGDB_KEYS[session_id] = generate_encryption_key()
    index_name = f"session_{session_id}"
    # CyborgVectorStore creates indexes without a metadata schema, so create the
    # index here with `_content` (where the wrapper stores each chunk's text)
    # marked full_text to enable BM25. The wrapper then loads the existing index.
    cyborgdb.Client(base_url=CYBORGDB_HOST, api_key=None).create_index(
        index_name=index_name,
        index_key=CYBORGDB_KEYS[session_id],
        dimension=EMBEDDING_DIM,
        metric="cosine",
        metadata_schema={"_content": {"full_text": True}},
    )
    vector_store = CyborgVectorStore(
        index_name=index_name,
        index_key=CYBORGDB_KEYS[session_id],
        api_key=None,
        base_url=CYBORGDB_HOST,
        embedding=EMBEDDING_MODEL_ID,
        metric="cosine",
    )
    VECTORSTORE_STORE[session_id] = vector_store
    return vector_store

def upload_documents(file_paths: list[str], session_id: str) -> str:
    try:
        if not file_paths:
            return "❌ No files selected"

        print(f"Processing {len(file_paths)} files for session {session_id}")
        vectorstore = load_vectorstore(session_id)
        encryption_key = bytes(CYBORGDB_KEYS[session_id])
        all_docs = []
        previews = []

        for file_path in file_paths:
            print(f"Processing: {file_path}")
            docs = split_documents(file_path)
            all_docs.extend(docs)
            for i, doc in enumerate(docs[:3]):
                encrypted = encrypt_chunk(doc.page_content, encryption_key)
                previews.append(f"  Chunk {i+1}: {encrypted[:60]}...")

        if all_docs:
            print(f"Adding {len(all_docs)} documents to vector store")
            vectorstore.add_documents(all_docs)
            print("Training index...")
            vectorstore.index.train()
            print("Index trained successfully")

        result = [
            f"✅ Processed {len(file_paths)} file(s) with {len(all_docs)} chunks",
            "",
            "🔐 Encrypted chunk previews:"
        ]
        result.extend(previews[:5])
        if len(all_docs) > 5:
            result.append(f"  ... and {len(all_docs) - 5} more encrypted chunks")
        return "\n".join(result)
    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"ERROR in upload_documents: {error_msg}")
        return f"❌ Error: {str(e)}\n\nSee logs for details."

def retrieve(vectorstore: CyborgVectorStore, query: str, mode: str, k: int = 3) -> list[str]:
    results = vectorstore.index.query(
        query_vectors=vectorstore.get_embeddings(query).tolist(),
        text=query,
        alpha=RETRIEVAL_MODES[mode],
        top_k=k,
        include=["metadata"],
    )
    return [r["metadata"]["_content"] for r in results]

def get_llm():
    return langchain_openai.ChatOpenAI(
        base_url=OPENAI_URL,
        model=OPENAI_MODEL,
        max_tokens=500,
        temperature=0.7,
    )

def create_comparison_html(query: str, result_docs=None, plaintext_embeddings=None, encrypted_embeddings=None, mode=None):
    """Create HTML showing plaintext vs encrypted vectors."""
    if not result_docs or not plaintext_embeddings:
        return f"""
        <div style="margin: 20px 0; border: 2px solid #e2e8f0; border-radius: 12px; overflow: hidden; background: white; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
            <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; text-align: center;">
                <h2 style="color: white; margin: 0; font-size: 24px;">🔒 Retrieved Results: Plaintext vs Encrypted Vectors</h2>
            </div>
            <div style="padding: 40px; text-align: center; color: #666;">
                <p style="font-size: 16px;">Query: "<strong>{query}</strong>"</p>
                <p style="margin-top: 16px; color: #e53e3e; font-weight: 600;">⚠️ No documents found. Please upload documents first!</p>
            </div>
        </div>
        """

    results_html = ""
    for i, (doc, plaintext_emb, encrypted_emb) in enumerate(zip(result_docs, plaintext_embeddings, encrypted_embeddings)):
        content_preview = doc[:150] if len(doc) > 150 else doc
        plaintext_preview = "[" + ", ".join([f"{x:.4f}" for x in plaintext_emb[:20]]) + ", ...]"
        encrypted_preview = encrypted_emb[:150] + "..." if len(encrypted_emb) > 150 else encrypted_emb

        results_html += f"""
        <div style="margin-bottom: 16px; padding: 14px; background: #fafafa; border-radius: 8px; border: 1px solid #e2e8f0;">
            <div style="font-weight: 600; color: #333; margin-bottom: 10px; font-size: 15px;">
                📄 Result #{i+1}
            </div>
            <div style="margin-bottom: 12px; padding: 10px; background: white; border-radius: 6px; border: 1px solid #e2e8f0;">
                <div style="font-size: 14px; color: #666; font-weight: 600; margin-bottom: 4px;">
                    📝 Document Content:
                </div>
                <div style="font-size: 14px; color: #2d3748; line-height: 1.5;">
                    "{content_preview}..."
                </div>
            </div>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
                <div style="padding: 10px; background: #fff5f5; border: 2px solid #fc8181; border-radius: 6px;">
                    <div style="font-size: 14px; color: #c53030; font-weight: 600; margin-bottom: 6px;">
                        🔓 Plaintext Vector (first 20 dims):
                    </div>
                    <div style="background: white; padding: 6px; border-radius: 4px; font-family: 'Courier New', monospace; font-size: 14px; color: #4a5568; max-height: 120px; overflow-y: auto; line-height: 1.6; word-break: break-all;">
                        {plaintext_preview}
                    </div>
                </div>
                <div style="padding: 10px; background: #f0fff4; border: 2px solid #9ae6b4; border-radius: 6px;">
                    <div style="font-size: 14px; color: #22543d; font-weight: 600; margin-bottom: 6px;">
                        🔒 Encrypted Vector (first 150 chars):
                    </div>
                    <div style="background: white; padding: 6px; border-radius: 4px; font-family: 'Courier New', monospace; font-size: 14px; color: #4a5568; max-height: 120px; overflow-y: auto; word-break: break-all; line-height: 1.6;">
                        {encrypted_preview}
                    </div>
                </div>
            </div>
        </div>
        """

    return f"""
    <div style="--body-text-color: #1a202c; margin: 20px 0; border: 2px solid #e2e8f0; border-radius: 12px; overflow: hidden; background: white; 
  box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; text-align: center;">
            <h2 style="color: white; margin: 0; font-size: 24px;">🔒 Retrieved Results: Plaintext vs Encrypted Vectors</h2>
        </div>
        <div style="padding: 16px; background: #f9fafb;">
            <div style="font-size: 15px; color: #666; margin-bottom: 16px; padding: 8px; background: white; border-radius: 6px;">
                <strong>Your Query:</strong> "{query}" &nbsp;·&nbsp; <strong>Retrieval:</strong> {mode}
            </div>
            <div style="margin-bottom: 16px; padding: 16px; background: #edf2f7; border-radius: 8px;">
                <div style="text-align: center; margin-bottom: 12px;">
                    <p style="margin: 0; color: #2d3748; font-size: 16px; font-weight: 600;">
                        💡 Why Encrypt Embeddings?
                    </p>
                </div>
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
                    <div style="background: white; padding: 12px; border-radius: 6px; border-left: 4px solid #fc8181;">
                        <div style="font-size: 14px; color: #c53030; font-weight: 600; margin-bottom: 8px;">
                            🚨 The Problem
                        </div>
                        <ul style="margin: 0; padding-left: 20px; font-size: 14px; color: #1a202c; line-height: 1.8;">
                            <li>Embeddings capture semantic meaning of your documents</li>
                            <li>Machine learning attacks can reconstruct original text</li>
                            <li>Database breaches expose sensitive information</li>
                            <li>Compliance regulations require data protection</li>
                        </ul>
                    </div>
                    <div style="background: white; padding: 12px; border-radius: 6px; border-left: 4px solid #9ae6b4;">
                        <div style="font-size: 14px; color: #22543d; font-weight: 600; margin-bottom: 8px;">
                            ✅ The Solution
                        </div>
                        <ul style="margin: 0; padding-left: 20px; font-size: 14px; color: #1a202c; line-height: 1.8;">
                            <li>CyborgDB encrypts vectors before storing them</li>
                            <li>Homomorphic encryption enables search on encrypted data</li>
                            <li>Breaches only expose encrypted, meaningless data</li>
                            <li>Maintains full RAG functionality with zero trust</li>
                        </ul>
                    </div>
                </div>
            </div>
            {results_html}
        </div>
    </div>
    """

# Matches the 33-word Lucene English stop set CyborgDB's BM25 analyzer drops
# (cyborgdb-core src/text/stopwords.hpp); every other query word is searchable.
BM25_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "if", "in", "into", "is", "it",
    "no", "not", "of", "on", "or", "such", "that", "the", "their", "then", "there", "these",
    "they", "this", "to", "was", "will", "with",
}

def query_terms(query: str) -> list[str]:
    words = re.findall(r"[a-z0-9]+", query.lower())
    return [w for w in dict.fromkeys(words) if w not in BM25_STOPWORDS]

def highlight_terms(text: str, terms: list[str]) -> str:
    escaped = html.escape(text)
    if not terms:
        return escaped
    # Longer terms also match as prefixes so stemmed BM25 hits ("walk" -> "walked") are
    # highlighted; shorter ones only match whole words.
    alternatives = [re.escape(t) + (r"\w*" if len(t) >= 4 else "") for t in terms]
    pattern = re.compile(r"\b(?:" + "|".join(alternatives) + r")\b", re.IGNORECASE)
    return pattern.sub(
        lambda m: f'<mark style="background: #fefcbf; color: #1a202c; padding: 0 2px; border-radius: 3px;">{m.group(0)}</mark>',
        escaped,
    )

def create_retrieval_comparison_html(query: str, results_by_mode=None, selected_mode=None):
    """Show the top results from every retrieval mode side by side."""
    if not results_by_mode:
        body = """
            <div style="padding: 40px; text-align: center; color: #666;">
                <p style="font-size: 16px;">Ask a question to compare what each retrieval mode finds.</p>
            </div>
        """
    else:
        terms = query_terms(query)
        columns = ""
        for mode, chunks in results_by_mode.items():
            found_elsewhere = {c for m, cs in results_by_mode.items() if m != mode for c in cs}
            cards = ""
            for rank, chunk in enumerate(chunks, 1):
                badge = "" if chunk in found_elsewhere else f"""
                    <span style="margin-left: 6px; padding: 2px 8px; background: #fbd38d; color: #744210; border-radius: 999px; font-size: 12px; font-weight: 600;">only {mode}</span>
                """
                cards += f"""
                <div style="margin-top: 10px; padding: 10px; background: white; border: 1px solid #e2e8f0; border-radius: 6px;">
                    <div style="font-size: 13px; font-weight: 600; color: #4a5568; margin-bottom: 6px;">#{rank}{badge}</div>
                    <div style="font-size: 13px; color: #2d3748; line-height: 1.5; max-height: 220px; overflow-y: auto; white-space: pre-wrap;">{highlight_terms(chunk, terms)}</div>
                </div>
                """
            if not chunks:
                cards = """
                <div style="margin-top: 10px; padding: 10px; color: #718096; font-size: 13px; font-style: italic;">No matching chunks</div>
                """
            selected = mode == selected_mode
            border = "#667eea" if selected else "#e2e8f0"
            sent = """<span style="margin-left: 6px; font-size: 12px; color: #667eea;">→ sent to LLM</span>""" if selected else ""
            columns += f"""
            <div style="padding: 12px; background: #f7fafc; border: 2px solid {border}; border-radius: 8px; min-width: 0;">
                <div style="font-size: 16px; font-weight: 700; color: #2d3748;">{mode}{sent}</div>
                <div style="font-size: 12px; color: #718096;">alpha = {RETRIEVAL_MODES[mode]}</div>
                {cards}
            </div>
            """
        body = f"""
            <div style="padding: 16px; background: #f9fafb;">
                <div style="font-size: 14px; color: #4a5568; margin-bottom: 12px; padding: 8px; background: white; border-radius: 6px; line-height: 1.6;">
                    <strong>Your Query:</strong> "{html.escape(query)}"<br>
                    <mark style="background: #fefcbf; color: #1a202c; padding: 0 2px; border-radius: 3px;">Highlighted</mark> words are query terms BM25 can match.
                    <span style="padding: 1px 6px; background: #fbd38d; color: #744210; border-radius: 999px; font-size: 12px; font-weight: 600;">only …</span> marks a chunk no other mode retrieved.
                </div>
                <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 12px;">
                    {columns}
                </div>
            </div>
        """

    return f"""
    <div style="--body-text-color: #1a202c; margin: 20px 0; border: 2px solid #e2e8f0; border-radius: 12px; overflow: hidden; background: white; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; text-align: center;">
            <h2 style="color: white; margin: 0; font-size: 24px;">⚖️ Retrieval Comparison: Vector vs Hybrid vs BM25</h2>
        </div>
        {body}
    </div>
    """

def chat(message: str, history, session: str, system_prompt: str, retrieval_mode: str) -> tuple[str, str, str]:
    global MOST_RECENT_PROMPT, LAST_QUERY_INFO
    MOST_RECENT_PROMPT = message

    try:
        print(f"\n=== Chat Query ===")
        print(f"Session: {session}")
        print(f"Message: {message}")
        print(f"Retrieval mode: {retrieval_mode}")

        CHAT_HISTORY_STORE.setdefault(session, [])
        if session not in LLM_STORE:
            LLM_STORE[session] = get_llm()
        llm = LLM_STORE[session]

        vectorstore = load_vectorstore(session)
        encryption_key = bytes(CYBORGDB_KEYS[session])

        # Retrieve documents
        print("Retrieving documents...")
        results_by_mode = {mode: retrieve(vectorstore, message, mode) for mode in RETRIEVAL_MODES}
        result_contents = results_by_mode[retrieval_mode]
        print(f"Retrieved {len(result_contents)} documents")

        # Process retrieved documents for display
        plaintext_embeddings = []
        encrypted_embeddings = []

        for content in result_contents:
            doc_plaintext_embedding = vectorstore.get_embeddings(content)
            plaintext_embeddings.append(doc_plaintext_embedding.tolist())
            doc_encrypted_embedding = encrypt_chunk(str(doc_plaintext_embedding.tolist()), encryption_key)
            encrypted_embeddings.append(doc_encrypted_embedding)

        LAST_QUERY_INFO = {
            'query': message,
            'result_contents': result_contents,
            'plaintext_embeddings': plaintext_embeddings,
            'encrypted_embeddings': encrypted_embeddings,
        }

        # Inline retrieved context into the system prompt (replaces the old {context} placeholder)
        context = "\n\n".join(result_contents) if result_contents else "(no documents retrieved)"
        system_with_context = system_prompt.replace("{context}", context)

        # Build message list: system + prior history + new user turn
        history_msgs = CHAT_HISTORY_STORE[session]
        messages = [("system", system_with_context), *history_msgs, ("human", message)]

        print("Invoking LLM...")
        response = llm.invoke(messages)
        answer = response.content
        print(f"Answer: {answer[:200]}...")

        # Persist this turn into history for next call
        history_msgs.append(("human", message))
        history_msgs.append(("ai", answer))

        comparison_html = create_comparison_html(
            message, result_contents, plaintext_embeddings, encrypted_embeddings, retrieval_mode
        )

        retrieval_html = create_retrieval_comparison_html(message, results_by_mode, retrieval_mode)

        return answer, comparison_html, retrieval_html

    except Exception as e:
        import traceback
        error_msg = traceback.format_exc()
        print(f"ERROR in chat: {error_msg}")
        error_html = create_comparison_html(message, None, None, None)
        return f"❌ Error: {str(e)}", error_html, create_retrieval_comparison_html(message)

print("✅ Functions defined")

# Gradio UI
def get_session(request: gr.Request) -> str:
    return request.session_hash

with gr.Blocks(title="🔒 Encrypted RAG Chatbot") as demo:
    gr.Markdown("# 🔒 Encrypted RAG Chatbot with CyborgDB")
    gr.Markdown("Upload documents and ask questions - see plaintext vs encrypted vectors from search results!")

    session = gr.State(value=str(uuid.uuid4()))

    with gr.Row():
        with gr.Column(scale=2, variant="panel"):
            gr.Markdown("## 💬 Chat with Your Documents")
            system_prompt = gr.Textbox(
                label="System instruction",
                lines=2,
                value="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Keep the answer concise. {context}",
                visible=False
            )

            chatbot = gr.Chatbot(height=400, label="Conversation")

            with gr.Row():
                msg = gr.Textbox(
                    label="Your question",
                    placeholder="Ask a question about your documents...",
                    scale=4,
                    show_label=False
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)

            with gr.Row():
                retrieval_mode = gr.Radio(
                    choices=list(RETRIEVAL_MODES),
                    value="Hybrid",
                    label="Retrieval mode",
                    info="Hybrid fuses BM25 keyword search with vector search. BM25 matches exact words, so distinctive terms like names and IDs work best.",
                    scale=4,
                )
                clear_btn = gr.Button("Clear Chat", size="sm", scale=1)

        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("## 📄 Upload Documents")
            file_input = gr.File(
                type="filepath",
                file_count="multiple",
                label="Select PDF or TXT files"
            )
            upload_btn = gr.Button(
                "🔒 Create Encrypted Vector Store",
                variant="primary",
                size="lg"
            )
            upload_output = gr.Textbox(
                show_label=False,
                lines=6,
                placeholder="Upload files and click the button above..."
            )

            upload_btn.click(
                fn=lambda files, sess: upload_documents(files if files else [], sess),
                inputs=[file_input, session],
                outputs=[upload_output]
            )

            gr.Markdown("---")
            gr.Markdown("""
            ### 📋 Quick Start
            1. Upload documents
            2. Create encrypted vector store
            3. Pick a retrieval mode and ask questions
            4. Compare retrieval modes and vectors below
            """)

    gr.Markdown("---")
    gr.Markdown("## ⚖️ Retrieval Comparison")
    gr.Markdown("*Each question runs through every retrieval mode; the selected mode's results are sent to the LLM*")

    retrieval_display = gr.HTML(
        create_retrieval_comparison_html("Your query will appear here"),
        label="Retrieval Comparison"
    )

    gr.Markdown("---")
    gr.Markdown("## 🔍 Retrieved Results: Vector Comparison")
    gr.Markdown("*Compare plaintext vs encrypted vectors from search results*")

    comparison_display = gr.HTML(
        create_comparison_html("Your query will appear here", None, None, None),
        label="Vector Comparison"
    )

    # Wire up chat (Gradio 6.x Chatbot uses messages format: list of {"role", "content"} dicts)
    def respond(message, history, sess, sys_prompt, mode):
        answer, updated_comparison, updated_retrieval = chat(message, history, sess, sys_prompt, mode)
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer},
        ]
        return "", history, updated_comparison, updated_retrieval

    msg.submit(
        respond,
        inputs=[msg, chatbot, session, system_prompt, retrieval_mode],
        outputs=[msg, chatbot, comparison_display, retrieval_display]
    )

    submit_btn.click(
        respond,
        inputs=[msg, chatbot, session, system_prompt, retrieval_mode],
        outputs=[msg, chatbot, comparison_display, retrieval_display]
    )

    clear_btn.click(lambda: [], None, chatbot)
    demo.load(get_session, None, session)

print("✅ Gradio interface created")

In [ ]:
# 4. Launch Gradio App

print("\n🚀 Launching Gradio app...")
demo.launch(share=True, debug=True, inline=False)
print("\n🎉 Chatbot is running! Click the public URL above.")